# NumPy

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*X. Cid Vidal* (original material, 2025). Adapted by *J. A. Hernando Morata*.

Python loops are slow; numerical physics runs on **arrays**. NumPy provides the array
and the fast operations on it. The algorithms built on top of it are in the next
chapter, [SciPy](./scipy.ipynb).

**Objectives**

* create, index and slice arrays, and know when you get a **view** or a **copy**;
* replace loops by **broadcasting** and **vectorized** operations;
* write matrix products and contractions with `@` and `einsum`.

In [ ]:
import timeit
import numpy as np
import matplotlib.pyplot as plt

## 1. NumPy arrays

A NumPy **array** (`np.ndarray`) is a grid of values of the same type (`dtype`), stored
contiguously in memory. Operations act on the whole array, in compiled code.

### 1.1 Creation, shape and type

In [ ]:
a = np.array([[1, 2, 3], [4, 5, 6]], dtype=float)
print(a)
print('shape:', a.shape, ' ndim:', a.ndim, ' dtype:', a.dtype)

In [ ]:
print(np.arange(0, 10, 2))     # start, stop (excluded), step
print(np.linspace(0, 1, 5))    # start, stop (included), number of points
print(np.zeros((2, 3)))

`arange` for integer steps; `linspace` for a fixed number of points (no rounding
surprises with float steps). More recipes in the notebook `cookbook_numpy.ipynb`.

### 1.2 Indexing and slicing

* `m[i, j]`: the element in row `i`, column `j`;
* `m[start:stop:step]` along each axis; `stop` is excluded, negative indices count
  from the end, an empty field means "all".

In [ ]:
m = np.arange(1, 13).reshape(3, 4)
print(m)
print('m[1, 2] =', m[1, 2])
print('first two rows:\n', m[:2, :])
print('last two columns:\n', m[:, -2:])

In [ ]:
img = np.arange(100).reshape(10, 10)
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(6, 3))
ax0.imshow(img, vmin=0, vmax=99)
ax0.add_patch(plt.Rectangle((2.5, 1.5), 5, 5, fill=False, ec='red', lw=2))
ax0.set_title('img')
ax1.imshow(img[2:7, 3:8], vmin=0, vmax=99)
ax1.set_title('img[2:7, 3:8]')
plt.show()

### 1.3 Boolean masks and fancy indexing

* A **boolean array** (a *mask*) selects the elements where it is `True`.
* A **list or array of integers** selects those positions (*fancy indexing*).

In [ ]:
n = np.arange(20)
mask = n % 3 == 0
print(mask[:7])
print('n[mask]      =', n[mask])
print('n[[0, 3, 5]] =', n[[0, 3, 5]])

In [ ]:
print(n[(n > 5) & (n % 2 == 0)])    # combine with &, |, ~ (and parentheses)

### 1.4 Views and copies

* A **slice** is a **view**: a new window on the same data, nothing is copied.
* **Fancy** and **boolean** indexing return a **copy**.
* `.copy()` forces an independent array.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — View or copy?</b></p>

**Decide** what `print(b)` shows, before running it.

```python
b = np.arange(6)
s = b[1:4]
f = b[[1, 2, 3]]
s[0] = -1
f[1] = -2
print(b)
```

</div>

In [ ]:
b = np.arange(6)
s = b[1:4]
f = b[[1, 2, 3]]
s[0] = -1
f[1] = -2
print(b)
print(np.shares_memory(b, s), np.shares_memory(b, f))

<details>
<summary><b>Solution</b></summary>

`[ 0 -1  2  3  4  5]`. `s` is a view: `s[0]` **is** `b[1]`. `f` is a copy: changing it
does not touch `b`. `np.shares_memory` confirms it.

Views make slicing cheap, but beware: unlike a list slice (a copy), a NumPy slice is a
view. A function that modifies a slice of its argument modifies the caller's array.

</details>

## 2. Broadcasting

Operations between arrays of **different shapes**, without loops and without copies.

* Compare the shapes **from the right**; a missing dimension counts as 1.
* Two dimensions are compatible if they are **equal**, or **one of them is 1**.
* The dimensions of size 1 are stretched to match the other array.
* `np.newaxis` (an alias of `None`) inserts an axis of length 1:
  `v[:, np.newaxis]` turns a `(n,)` vector into a `(n, 1)` column.

In [ ]:
row = np.array([1, 2, 3])              # shape (3,)
col = np.array([[10], [20], [30]])     # shape (3, 1)
print(row + col)
print((row + col).shape)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which shape?</b></p>

**Predict** the shape of each result, or whether it fails.

```python
np.ones((4, 3)) + np.arange(3)
np.ones((4, 1)) + np.arange(3)
np.ones((2, 1, 3)) + np.ones((4, 1))
np.ones((4, 3)) + np.arange(4)
```

</div>

In [ ]:
print((np.ones((4, 3)) + np.arange(3)).shape)
print((np.ones((4, 1)) + np.arange(3)).shape)
print((np.ones((2, 1, 3)) + np.ones((4, 1))).shape)
np.ones((4, 3)) + np.arange(4)

<details>
<summary><b>Solution</b></summary>

`(4, 3)`, `(4, 3)`, `(2, 4, 3)` and a `ValueError`: `(4, 3)` and `(4,)` are compared
from the right, `3` against `4`. To add a vector to each **column**, give it the shape
`(4, 1)`: `np.arange(4)[:, np.newaxis]`.

</details>

### 2.1 Example: centering data

Subtract the mean of each column, `(3, 4) - (4,)`. For the mean of each row, keep the
dimension, so that the shape is `(3, 1)`:

In [ ]:
data = np.arange(12).reshape(3, 4)
col_mean = data.mean(axis=0)                  # shape (4,)
row_mean = data.mean(axis=1, keepdims=True)   # shape (3, 1)
print(data)
print(col_mean)
print(row_mean)
print(data - col_mean)
print(data - row_mean)

### 2.2 Functions on a grid

To evaluate $f(x, y)$ on a grid, `np.meshgrid` builds the 2D arrays of coordinates. The
same result comes from broadcasting a row of $x$ and a column of $y$:

In [ ]:
x = np.linspace(-2, 2, 5)
y = np.linspace(-1, 1, 3)
X, Y = np.meshgrid(x, y)          # rows follow y, columns follow x
print(X.shape, Y.shape)
print("X = ", X)
print("Y = ", Y)

r2_mesh = X**2 + Y**2
r2_bcast = x[np.newaxis, :]**2 + y[:, np.newaxis]**2
print(np.array_equal(r2_mesh, r2_bcast))

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A 2D Gaussian</b></p>

Write `gaussian2d(x, y, x0, y0, sx, sy, rho)`, the (unnormalized) bivariate Gaussian

$$
g = \exp\left[-\frac{u^2 + v^2 - 2\rho\, u v}{2(1-\rho^2)}\right], \qquad
u = \frac{x - x_0}{\sigma_x}, \quad v = \frac{y - y_0}{\sigma_y}.
$$

* Unnormalized ($g = 1$ at the center); valid for $|\rho| < 1$.

1. Evaluate it on a grid of $151 \times 101$ points in $[-3, 3] \times [-2, 2]$, with
   $x_0 = 0.6$, $y_0 = -0.2$ (both on the grid), $\sigma_x = 0.8$, $\sigma_y = 0.5$.
2. Draw it with `plt.contourf` for $\rho = 0$ and $\rho = 0.6$.
3. Check that the maximum of the grid is 1, at $(x_0, y_0)$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def gaussian2d(x, y, x0=0., y0=0., sx=1., sy=1., rho=0.):
    u, v = (x - x0) / sx, (y - y0) / sy
    return np.exp(-(u**2 + v**2 - 2 * rho * u * v) / (2 * (1 - rho**2)))

xg, yg = np.meshgrid(np.linspace(-3, 3, 151), np.linspace(-2, 2, 101))
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, rho in zip(axs, [0., 0.6]):
    g = gaussian2d(xg, yg, x0=0.6, y0=-0.2, sx=0.8, sy=0.5, rho=rho)
    ax.contourf(xg, yg, g, levels=20)
    ax.set_title(f'rho = {rho}')
    i, j = np.unravel_index(np.argmax(g), g.shape)
    print(f'rho = {rho}: max {g.max():.3f} at ({xg[i, j]:.2f}, {yg[i, j]:.2f})')
plt.show()
```

* No loops: it works for numbers, 1D arrays and the 2D grids of `meshgrid`.
* The tilt comes from the term $-2\rho\, uv$ (correlation between $x$ and $y$).
* The factor $1/(1-\rho^2)$ keeps the marginal widths equal to $\sigma_x$ and $\sigma_y$.
* With a center off the grid, the grid maximum is slightly below 1.

</details>

### 2.3 Images: `imshow` and the axes

`plt.imshow(Z)` draws an array the way a matrix is written:

* row `i` on the vertical axis, **downwards**: row 0 is at the top;
* column `j` on the horizontal axis;
* the ticks are indices, not the values of $x$ and $y$.

A grid from `meshgrid` has $y$ growing with the row index: by default the picture is
**upside down**. The fix: `origin='lower'` and `extent=[xmin, xmax, ymin, ymax]`.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Where is the peak?</b></p>

$Z$ has its peak at $x = 1.5$, $y = 1$, in the **upper right** of the domain
$[-3, 3] \times [-2, 2]$. Where does it appear with a plain `imshow(Z)`?

</div>

In [ ]:
x = np.linspace(-3, 3, 121)
y = np.linspace(-2, 2, 81)
X, Y = np.meshgrid(x, y)                        # shape (81, 121) = (ny, nx)
Z = np.exp(-((X - 1.5)**2 + (Y - 1)**2))       # peak at (1.5, 1)

fig, axs = plt.subplots(1, 3, figsize=(11, 3))
axs[0].imshow(Z)
axs[0].set_title('imshow(Z)')
axs[1].imshow(Z, origin='lower', extent=[x[0], x[-1], y[0], y[-1]])
axs[1].set_title("origin='lower', extent")
axs[2].contourf(X, Y, Z, levels=20)
axs[2].set_title('contourf(X, Y, Z)')
axs[2].set_aspect('equal')
for ax in axs[1:]:
    ax.plot(1.5, 1, 'r+', ms=12)                  # the true peak
plt.show()

<details>
<summary><b>Solution</b></summary>

In the **lower right**, around pixel (row 60, column 90): `imshow` puts row 0 ($y = -2$)
on top. `contourf(X, Y, Z)` uses the coordinates, so it needs no fix.

</details>

#### Two conventions for a grid

* `meshgrid(x, y)` (default, `indexing='xy'`): shape `(ny, nx)`, rows follow $y$. This
  is what `imshow` and `contourf` expect.
* `meshgrid(x, y, indexing='ij')`: shape `(nx, ny)`, `Z[i, j] = f(x[i], y[j])`. Natural
  in simulations, but it must be **transposed** for `imshow`.
* `np.histogram2d(x, y)` also returns `H[i, j]` with `i` the bin in $x$:
  `imshow(H.T, origin='lower', extent=...)`.

In [ ]:
Xij, Yij = np.meshgrid(x, y, indexing='ij')
Zij = np.exp(-((Xij - 1.5)**2 + (Yij - 1)**2))
print(Z.shape, Zij.shape, np.array_equal(Zij.T, Z))

***[+] Lookout.*** `extent` gives the **edges** of the image, while `x` and `y` are the
**centers** of the pixels: with `extent=[x[0], x[-1], ...]` the picture is shifted by half
a pixel. For an exact match, widen the extent by $\Delta x/2$ and $\Delta y/2$, or use
`plt.pcolormesh(x, y, Z)`, which takes the coordinates directly.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A 2D histogram</b></p>

Generate $10^5$ points from a 2D Gaussian with `rng = np.random.default_rng(7)`:

```python
pts = rng.multivariate_normal([1, 0.5], [[1, 0.6], [0.6, 0.5]], size=100_000)
```

1. Fill `H, xe, ye = np.histogram2d(...)` with $60 \times 40$ bins in
   $[-3, 5] \times [-2, 3]$. Check `H.shape` and `H.sum()`.
2. Draw `H` with `imshow`, with the axes right. Compare with `plt.hist2d`.
3. Without loops, compute the **profile**: the mean of $y$ in each bin of $x$. Draw it on
   top of the image. Its slope should be $\mathrm{cov}(x, y)/\sigma_x^2 = 0.6$.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
rng = np.random.default_rng(7)
pts = rng.multivariate_normal([1, 0.5], [[1, 0.6], [0.6, 0.5]], size=100_000)
H, xe, ye = np.histogram2d(pts[:, 0], pts[:, 1], bins=(60, 40),
                           range=[[-3, 5], [-2, 3]])
print(H.shape, H.sum())                       # (60, 40): (nx, ny)

xc = 0.5 * (xe[1:] + xe[:-1])                 # bin centers
yc = 0.5 * (ye[1:] + ye[:-1])
n = H.sum(axis=1)                             # entries per x-bin
ok = n > 0                                    # avoid 0/0 in empty bins
prof = np.full(len(xc), np.nan)
prof[ok] = (H[ok] * yc).sum(axis=1) / n[ok]   # (nx, ny) * (ny,): broadcasting

fig, axs = plt.subplots(1, 2, figsize=(9, 3))
axs[0].imshow(H.T, origin='lower', extent=[xe[0], xe[-1], ye[0], ye[-1]],
              aspect='auto')
axs[0].plot(xc, prof, 'r.', ms=4)
axs[0].set_title("imshow(H.T, origin='lower')")
axs[1].hist2d(pts[:, 0], pts[:, 1], bins=(60, 40), range=[[-3, 5], [-2, 3]])
axs[1].set_title('hist2d')
plt.show()

sel = n > 100                                 # fit only well-filled bins
print('slope:', np.polyfit(xc[sel], prof[sel], 1)[0])
```

* `H.shape` is `(60, 40)`: the first index is $x$. Hence the `.T`.
* `H.sum()` is $10^5$ minus the few points outside the range.
* The profile is a weighted mean along `axis=1`; the slope is $\approx 0.6$.

</details>

## 3. Vectorization

A Python loop runs one interpreted step per element; a NumPy function runs the whole loop
in compiled code. The rule: **no explicit loops over the elements of an array**.

In [ ]:
def loop_row_sum(x):
    """Sum of each row of a 2D array, with Python loops."""
    out = np.empty(x.shape[0])
    for i in range(x.shape[0]):
        s = 0.
        for j in range(x.shape[1]):
            s += x[i, j]
        out[i] = s
    return out

rng = np.random.default_rng(1)
xs = rng.random((2000, 50))
assert np.allclose(loop_row_sum(xs), xs.sum(axis=1))

In [ ]:
t_loop = timeit.timeit(lambda: loop_row_sum(xs), number=3) / 3
t_vec = timeit.timeit(lambda: xs.sum(axis=1), number=100) / 100
print(f'loop: {t_loop * 1e3:.1f} ms, numpy: {t_vec * 1e3:.3f} ms')

The times depend on the machine; the ratio is typically $10^2$–$10^3$.
In IPython, `%timeit xs.sum(axis=1)` does the same in one line.

### 3.1 Matrix products and `einsum`

* `A @ B` is the matrix product (`np.matmul`); `A * B` multiplies element by element.
* `np.einsum` writes any contraction in Einstein notation: `'ij,jk->ik'` means
  $C_{ik} = \sum_j A_{ij} B_{jk}$.

In [ ]:
rng = np.random.default_rng(2)
A = rng.random((3, 2))
B = rng.random((2, 4))
C = A @ B
print(A, B)
print(C)
print(C.shape, np.allclose(C, np.einsum('ij,jk->ik', A, B)))

In [ ]:
print(np.einsum('ii->', np.eye(3)))           # trace
print(A.shape, np.einsum('ij->ji', A).shape)           # transpose
print(np.einsum('ij,ij->i', A, A).round(3))   # squared norm of each row

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Distance matrix</b></p>

Given $N$ points in 3D, an array `p` of shape `(N, 3)`, compute the $N \times N$ matrix
of distances $d_{ij} = |\mathbf{p}_i - \mathbf{p}_j|$:

1. with two Python loops;
2. with broadcasting: `p[:, np.newaxis, :] - p[np.newaxis, :, :]` has shape `(N, N, 3)`.

Check that both agree. Time them for $N = 100, 200, 400$ and plot the time against $N$
in log-log scale. What slope do you expect?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
def dist_loop(p):
    n = len(p)
    d = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            d[i, j] = np.sqrt(np.sum((p[i] - p[j])**2))
    return d

def dist_bcast(p):
    diff = p[:, np.newaxis, :] - p[np.newaxis, :, :]    # (N, N, 3)
    return np.sqrt(np.sum(diff**2, axis=-1))

rng = np.random.default_rng(3)
ns = [100, 200, 400]
t_loops, t_bcasts = [], []
for n in ns:
    p = rng.random((n, 3))
    assert np.allclose(dist_loop(p), dist_bcast(p))
    t_loops.append(timeit.timeit(lambda: dist_loop(p), number=1))
    t_bcasts.append(timeit.timeit(lambda: dist_bcast(p), number=5) / 5)

for name, ts in [('loops', t_loops), ('broadcasting', t_bcasts)]:
    slope = np.polyfit(np.log(ns), np.log(ts), 1)[0]
    print(f'{name:12s}: slope {slope:.2f}')
print('ratio loop/bcast:', np.round(np.array(t_loops) / np.array(t_bcasts)))

plt.loglog(ns, t_loops, 'o-', label='loops')
plt.loglog(ns, t_bcasts, 's-', label='broadcasting')
plt.xlabel('N'); plt.ylabel('time (s)'); plt.legend()
plt.show()
```

* Expected slope 2: there are $N^2$ distances. The fitted slopes are close to 2 (for
  small $N$ the fixed overhead of each call lowers the broadcasting slope; it approaches
  2 for large $N$).
* Broadcasting only changes the constant: the ratio is typically $10^2$–$10^3$.
* Its cost is memory: the intermediate array has $3N^2$ numbers.
  `scipy.spatial.distance.cdist(p, p)` avoids it.

</details>

## 4. Working with data

Common operations on measured data, all without loops:

* join arrays; sort them;
* select with conditions, handle missing values;
* count; accumulate and differentiate; rebin.

### 4.1 Joining arrays

* `np.concatenate`: along an **existing** axis.
* `np.stack`: along a **new** axis; all arrays with the same shape.
* `np.column_stack`: 1D arrays as the columns of a table.

In [ ]:
a, b = np.arange(3), np.arange(3, 6)
print(np.concatenate([a, b]))          # (6,)
print(np.stack([a, b]))                # (2, 3): new axis 0
print(np.stack([a, b], axis=1))        # (3, 2): new axis 1
print(np.column_stack([a, b]).shape)   # (3, 2)

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Join two matrices</b></p>

`m = np.ones((2, 3))`. **Predict** the shapes.

```python
np.concatenate([m, m])
np.concatenate([m, m], axis=1)
np.stack([m, m])
np.stack([m, m], axis=-1)
```

</div>

In [ ]:
m = np.ones((2, 3))
print(np.concatenate([m, m]).shape)
print(np.concatenate([m, m], axis=1).shape)
print(np.stack([m, m]).shape)
print(np.stack([m, m], axis=-1).shape)

<details>
<summary><b>Solution</b></summary>

`(4, 3)`, `(2, 6)`, `(2, 2, 3)` and `(2, 3, 2)`. `concatenate` keeps the number of axes;
`stack` adds one, at the position given by `axis`.

</details>

### 4.2 Sorting

* `np.sort(a)`: a sorted copy.
* `np.argsort(a)`: the **indices** that sort `a`. Apply them to other arrays to sort a
  table by one of its columns.
* `np.argpartition(a, -k)[-k:]`: the `k` largest, without sorting everything.

In [ ]:
rng = np.random.default_rng(20)
E = rng.exponential(10, size=8).round(1)      # energies of 8 events
t = np.arange(8)                              # their times
order = np.argsort(E)[::-1]                   # decreasing energy
print(E[order])
print(t[order])                               # the times, in the same order
top3 = np.argpartition(E, -3)[-3:]            # the 3 largest, unordered
print(np.sort(E[top3]))

### 4.3 Conditions and missing values

* `np.where(cond, a, b)`: element-wise *if-else*. `np.where(cond)` alone: the indices.
* `np.nan` marks a missing value. It **propagates**: `x.mean()` is `nan`.
* `np.nanmean`, `np.nansum`, `np.nanmax`... skip the `nan`.
* `np.isnan(x)` finds them; `x == np.nan` is always `False`.

In [ ]:
x = np.array([1.2, -0.5, np.nan, 3.1, -2.0, np.nan])
print(np.where(x > 0, x, 0))       # careful: nan > 0 is False, so nan -> 0
print(np.where(x > 0))             # indices
print(x.mean(), np.nanmean(x))
print(np.isnan(x).sum(), (x == np.nan).sum())

### 4.4 Counting

* `np.unique(a, return_counts=True)`: the different values and how many times each.
* `np.bincount(k)`: for integers $\ge 0$, `counts[k]` is the number of times `k`
  appears. With `weights`, it sums the weights instead: a total per category.

In [ ]:
ch = np.array([3, 0, 3, 1, 3, 0, 4])            # channel hit by each particle
E = np.array([1., 2., 1.5, 0.5, 2., 1., 3.])    # its energy
vals, counts = np.unique(ch, return_counts=True)
print(vals, counts)
print(np.bincount(ch))                           # channels 0, 1, ..., 4
print(np.bincount(ch, weights=E))                # energy per channel

### 4.5 Cumulative sums and differences

* `np.cumsum`: running sum. A random walk, a discrete integral, a cumulative
  distribution.
* `np.diff`: differences of neighbours, $n - 1$ values. A derivative, the steps of a
  grid, the changes of sign.

In [ ]:
t = np.linspace(0, 2 * np.pi, 10)
y = np.cos(t)
dydt = np.diff(y) / np.diff(t)                  # forward derivative, 9 values
print(dydt.round(2))
cross = np.nonzero(np.diff(np.sign(y)))[0]      # where the sign changes
print('zero between', t[cross].round(2), 'and', t[cross + 1].round(2))

In [ ]:
rng = np.random.default_rng(21)
walk = np.cumsum(rng.choice([-1, 1], size=1000))
counts, edges = np.histogram(rng.normal(size=10_000), bins=40, range=(-4, 4))
cdf = np.cumsum(counts) / counts.sum()

fig, axs = plt.subplots(1, 2, figsize=(9, 3))
axs[0].plot(walk)
axs[0].set_title('random walk')
axs[1].step(edges[1:], cdf)
axs[1].set_title('cumulative distribution')
plt.show()

### 4.6 Rebinning with `reshape`

`a.reshape(-1, k)` puts each group of `k` consecutive values in a row; `.mean(axis=1)`
averages them. The size must be a multiple of `k` (trim the array first). In 2D, an
image `(ny, nx)` is rebinned with `.reshape(ny // k, k, nx // k, k).mean(axis=(1, 3))`.

In [ ]:
rng = np.random.default_rng(22)
t = np.arange(1000)
signal = np.sin(2 * np.pi * t / 500) + rng.normal(0, 0.5, t.size)

k = 50
blocks = signal.reshape(-1, k)                    # (20, 50)
s_mean = blocks.mean(axis=1)
s_err = blocks.std(axis=1, ddof=1) / np.sqrt(k)
t_mean = t.reshape(-1, k).mean(axis=1)

plt.plot(t, signal, '.', ms=2, alpha=0.3)
plt.errorbar(t_mean, s_mean, s_err, fmt='o')
plt.show()

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A detector run</b></p>

A run of 100 s records, for each event, its time, its channel (0–7) and its energy.
Some readouts failed: their energy is `nan`.

```python
rng = np.random.default_rng(23)
n = 5000
time = rng.uniform(0, 100, n)                 # s, not in order
chan = rng.integers(0, 8, n)
energy = rng.exponential(2.0, n)              # MeV
energy[rng.random(n) < 0.02] = np.nan         # failed readouts
```

1. Sort the three arrays by time.
2. Count the failed readouts and remove those events.
3. Number of events and total energy per channel.
4. Number of events per second (`np.histogram`), then the mean rate in bins of 10 s.
   Check with `np.cumsum` that no event is lost.
5. Put the result in a table of shape `(n, 3)` and check that its times are in order.

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```python
rng = np.random.default_rng(23)
n = 5000
time = rng.uniform(0, 100, n)
chan = rng.integers(0, 8, n)
energy = rng.exponential(2.0, n)
energy[rng.random(n) < 0.02] = np.nan

order = np.argsort(time)                                  # 1
time, chan, energy = time[order], chan[order], energy[order]

ok = ~np.isnan(energy)                                    # 2
print('failed:', (~ok).sum())
time, chan, energy = time[ok], chan[ok], energy[ok]

print('events:', np.bincount(chan, minlength=8))          # 3
print('energy:', np.bincount(chan, weights=energy, minlength=8).round(0))

counts, edges = np.histogram(time, bins=100, range=(0, 100))    # 4
rate10 = counts.reshape(-1, 10).mean(axis=1)
print('rate (1/s):', rate10)
print(np.cumsum(counts)[-1] == time.size)

table = np.column_stack([time, chan, energy])            # 5
print(table.shape, np.all(np.diff(table[:, 0]) >= 0))
```

* Sort first, with one `argsort` for the three arrays: they stay aligned.
* `minlength=8` keeps the 8 channels even if one has no events.
* The rate is $\approx 49$ events/s: $5000 \times 0.98 / 100$.
* `column_stack` turns the integer channels into floats: a table has one `dtype`.
  For mixed types, use pandas.

</details>

## 5. Summary

* Arrays, not loops: slicing (views), masks and fancy indexing (copies).
* Broadcasting: shapes are compared from the right; a dimension of 1 is stretched.
* `imshow` puts row 0 on top: use `origin='lower'` and `extent`.
* Data: `concatenate`/`stack`, `argsort`, `where` and `nan`-functions,
  `unique`/`bincount`, `cumsum`/`diff`, `reshape` to rebin.
* Vectorize: the loop runs in compiled code, typically $10^2$–$10^3$ times faster.

**Hints.** Before running, predict the shape of the result. Watch the memory of the
intermediate arrays that broadcasting creates.

<details>
<summary><b>[i] Cheat sheet — NumPy</b></summary>

| | |
|:--|:--|
| `np.array`, `arange`, `linspace`, `zeros` | create arrays |
| `a.shape`, `a.dtype`, `a.reshape` | shape and type |
| `a[i, j]`, `a[1:3, :]` | indexing, slicing (view) |
| `a[mask]`, `a[[0, 2]]` | boolean and fancy indexing (copy) |
| `a.copy()`, `np.shares_memory` | force a copy, check |
| `x[:, np.newaxis]`, `keepdims=True` | shapes for broadcasting |
| `np.meshgrid(x, y)` | coordinate grids, shape `(ny, nx)` |
| `imshow(Z, origin='lower', extent=...)` | image with the right axes |
| `np.histogram2d(x, y)` | 2D histogram, shape `(nx, ny)`: transpose it |
| `a.sum(axis=1)`, `a.mean(axis=0)` | reductions along an axis |
| `A @ B`, `np.einsum` | matrix product, contractions |
| `np.concatenate`, `np.stack`, `np.column_stack` | join arrays |
| `np.sort`, `np.argsort`, `np.argpartition` | sort, sorting indices, top k |
| `np.where(c, a, b)`, `np.isnan`, `np.nanmean` | conditions, missing values |
| `np.unique(..., return_counts=True)`, `np.bincount` | counting |
| `np.cumsum`, `np.diff` | running sum, differences |
| `a.reshape(-1, k).mean(axis=1)` | rebin by `k` |

**Traps**: a slice modifies the original; shapes aligned from the left;
`imshow` upside down; huge intermediate arrays; `nan` propagates and
`x == np.nan` is always `False`.

</details>

**Further reading**: [NumPy user guide](https://numpy.org/doc/stable/user/),
[Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/).

### Check yourself

1. `b = a[2:5]; b[:] = 0`. Has `a` changed? And with `b = a[[2, 3, 4]]`?
2. What is the shape of `np.ones((5, 1, 3)) * np.ones((4, 1))`?
3. Why is `xs.sum(axis=1)` much faster than a loop?
4. `H, xe, ye = np.histogram2d(x, y)`. What do you pass to `imshow`, and with what
   options?
5. `x` has a `nan`. What does `np.where(x > 0, x, 0)` put there?
6. Average 1000 samples in blocks of 50, without a loop.

<details>
<summary><b>Solutions</b></summary>

1. Yes, a slice is a view. No, fancy indexing makes a copy.
2. `(5, 4, 3)`.
3. `sum` runs its loop in compiled code; a Python loop pays one interpreter step per
   element.
4. `H.T`, with `origin='lower'` and `extent=[xe[0], xe[-1], ye[0], ye[-1]]`.
5. `0`: `nan > 0` is `False`. Deal with the `nan` first (`np.isnan`).
6. `a.reshape(-1, 50).mean(axis=1)`, of shape `(20,)`.

</details>